In [2]:
import pandas as pd

In [3]:
census = pd.read_csv('../data/usa_00001.csv')
cps = pd.read_csv('../data/cps_2000.csv')

In [ ]:
OCC90 = 'OCC1990'
AGE = 'AGE'
YOUNG = 'is_18_25'

In [5]:
print(cps.columns)

Index(['YEAR', 'SERIAL', 'MONTH', 'HWTFINL', 'CPSID', 'ASECFLAG', 'PERNUM',
       'WTFINL', 'CPSIDP', 'CPSIDV', 'AGE', 'OCC1990'],
      dtype='str')


In [8]:
census['is_18_25'] = census['AGE'].between(18,25)
cps['is_18_25'] = cps['AGE'].between(18,25)

In [11]:
census_occ = census.groupby(OCC90).agg(
    N=(OCC90, 'size'),
    np=(YOUNG, 'sum'))

census_occ['share'] = census_occ['np']/census_occ['N']

cps_occ = cps.groupby(OCC90).agg(
    N_CPS=(OCC90, 'size'),
    np_CPS=(YOUNG, 'sum'))

cps_occ['share_CPS'] = cps_occ['np_CPS']/cps_occ['N_CPS']


In [15]:
merged = census_occ.merge(cps_occ, on=OCC90, how='left', indicator=True)
merged = merged[merged['_merge'] != 'left_only']
merged = merged.drop(columns=['_merge'])

In [16]:
print(merged['share'].describe())
print(merged['share_CPS'].describe())

count    335.000000
mean       0.140397
std        0.088649
min        0.003460
25%        0.076364
50%        0.125504
75%        0.170356
max        0.519796
Name: share, dtype: float64
count    335.000000
mean       0.141685
std        0.098572
min        0.000000
25%        0.070514
50%        0.124180
75%        0.188823
max        0.680000
Name: share_CPS, dtype: float64


In [17]:
from scipy.stats import ttest_1samp

merged["error"] = merged["share"] - merged["share_CPS"]

ttest_1samp(merged["error"], 0)

TtestResult(statistic=np.float64(-0.5282494699706427), pvalue=np.float64(0.597676996538405), df=np.int64(334))

In [19]:
import statsmodels.api as sm

# OLS Regression: Census_Share = alpha + beta * CPS_Share
X = sm.add_constant(merged["share_CPS"])
model = sm.OLS(merged["share"], X).fit()

print(model.summary())
# Look for: Intercept ≈ 0, Slope ≈ 1, R-squared > 0.90

                            OLS Regression Results                            
Dep. Variable:                  share   R-squared:                       0.795
Model:                            OLS   Adj. R-squared:                  0.795
Method:                 Least Squares   F-statistic:                     1293.
Date:                Wed, 29 Jul 2026   Prob (F-statistic):          1.02e-116
Time:                        13:28:42   Log-Likelihood:                 602.53
No. Observations:                 335   AIC:                            -1201.
Df Residuals:                     333   BIC:                            -1193.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0268      0.004      6.957      0.0

In [22]:
merged_filtered = merged[merged['N_CPS'] >= 30]

# OLS Regression: Census_Share = alpha + beta * CPS_Share
X = sm.add_constant(merged_filtered["share_CPS"])
model = sm.OLS(merged_filtered["share"], X).fit()

print(model.summary())
# Look for: Intercept ≈ 0, Slope ≈ 1, R-squared > 0.90

                            OLS Regression Results                            
Dep. Variable:                  share   R-squared:                       0.795
Model:                            OLS   Adj. R-squared:                  0.795
Method:                 Least Squares   F-statistic:                     1286.
Date:                Wed, 29 Jul 2026   Prob (F-statistic):          4.72e-116
Time:                        13:30:46   Log-Likelihood:                 598.37
No. Observations:                 333   AIC:                            -1193.
Df Residuals:                     331   BIC:                            -1185.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0266      0.004      6.908      0.0